In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

cwd = os.getcwd()
root = cwd.split("BernsteinMartingaleNet")[0] + "BernsteinMartingaleNet"
if root not in sys.path:
    sys.path.append(root)

from lib.utils import get_sequence_data, train_model
from lib.BLogistic import BLogistic
from lib.DistHead import NormalHead, StudentTHead, SkewedStudentTHead

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

folder_path = root + r"/MarketData/historical_data"
context_window = 60
X, Y     = get_sequence_data(folder_path, context_window)
dof      = 16
simple_X = torch.tensor(X[:, :, 0], device=device)
simple_Y = torch.tensor(Y, device=device).reshape(-1, 1)

dev_size = 40000
test_size = 40000
np.random.seed(0)
indices = np.random.permutation(simple_X.shape[0])

dev_indices = indices[:dev_size]
test_indices = indices[dev_size:dev_size + test_size]
train_indices = indices[dev_size + test_size:]
train_X = simple_X[train_indices]
train_Y = simple_Y[train_indices]
dev_X = simple_X[dev_indices, :]
dev_Y = simple_Y[dev_indices]
test_X = simple_X[test_indices, :]
test_Y = simple_Y[test_indices]

std = train_Y.std()
train_X = train_X / std
train_Y = train_Y / std
dev_X = dev_X / std
dev_Y = dev_Y / std
test_X = test_X / std
test_Y = test_Y / std
print("train_X", train_X.shape, "train_Y", train_Y.shape, "dev_X", dev_X.shape, "dev_Y", dev_Y.shape)
print("std", std, train_X.std())

using cached data from C:\Users\MainUser\cs230\BernsteinMartingaleNet/MarketData/historical_data\spy_1min_data_context_60.npz
train_X torch.Size([332426, 60]) train_Y torch.Size([332426, 1]) dev_X torch.Size([40000, 60]) dev_Y torch.Size([40000, 1])
std tensor(0.0004, device='cuda:0') tensor(1.0205, device='cuda:0')


In [3]:

layer_sizes = [128, 64, 32]
#layer_sizes = [8, 8, 8] # small testing size

class LSTMProbNNDist(nn.Module):
    def __init__(self, context_window, dist_head, device):
        super().__init__()
        self.context_window = context_window
        self.dist_head      = dist_head
        self.device         = device

        print(f"\nInitializing LSTM with context_window={context_window}, dist={dist_head.__class__.__name__}, dof={dist_head.num_params()}")

        # LSTM layers (3 layers with decreasing neurons: 128 -> 64 -> 32)
        self.lstm1 = nn.LSTM(
            input_size=1,
            hidden_size=layer_sizes[0],
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.lstm2 = nn.LSTM(
            input_size=layer_sizes[0],
            hidden_size=layer_sizes[1],
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.lstm3 = nn.LSTM(
            input_size=layer_sizes[1],
            hidden_size=layer_sizes[2],
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.dropout = nn.Dropout(0.02)

        self.fc = nn.Linear(layer_sizes[2], dist_head.num_params())
        nn.init.uniform_(self.fc.weight, -0.01, 0.01)
        nn.init.zeros_(self.fc.bias)


    def forward(self, x, y):
        params = self.get_params(x)
        logpdf = self.dist_head.logpdf(
            y, params
        )
        return -logpdf.mean()

    def get_params(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() == 3 and x.shape[1] == 1:
            x = x.transpose(1, 2)

        x, _ = self.lstm1(x)
        x = self.dropout(x)

        x, _ = self.lstm2(x)
        x = self.dropout(x)

        x, _ = self.lstm3(x)
        x = self.dropout(x)

        x = x[:, -1, :]

        params = self.fc(x)
        return params

    def get_logpdf(self, x, sample_xs):
        params = self.get_params(x)
        return self.dist_head.logpdf(sample_xs, params)

    def get_pdf(self, x, sample_xs):
        return torch.exp(self.get_logpdf(x, sample_xs))

In [4]:
lr = 0.002
weight_decay = 0
num_steps = 300
batch_size = 512 * 8

In [4]:
model = LSTMProbNNDist(context_window, NormalHead(), device)
train_path = "Train_Normal/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)


Initializing LSTM with context_window=60, dist=NormalHead, dof=2
num batches 81
Init, Train Loss: 2.0924, Dev Loss: 2.0536, Dev Loss confidence interval: 2.0093, 2.0979
Step 0, Train Loss: 1.9182, Dev Loss: 1.8987, Dev Loss confidence interval: 1.8753, 1.9222
Step 10, Train Loss: 1.9177, Dev Loss: 1.8988, Dev Loss confidence interval: 1.8760, 1.9216
Step 20, Train Loss: 1.9185, Dev Loss: 1.8988, Dev Loss confidence interval: 1.8752, 1.9224
Step 30, Train Loss: 1.9176, Dev Loss: 1.8988, Dev Loss confidence interval: 1.8762, 1.9215
Step 40, Train Loss: 1.9176, Dev Loss: 1.8989, Dev Loss confidence interval: 1.8763, 1.9215
Step 50, Train Loss: 1.9176, Dev Loss: 1.8989, Dev Loss confidence interval: 1.8764, 1.9214
Step 60, Train Loss: 1.9176, Dev Loss: 1.8992, Dev Loss confidence interval: 1.8771, 1.9214
Step 70, Train Loss: 1.9176, Dev Loss: 1.8994, Dev Loss confidence interval: 1.8774, 1.9214
Step 80, Train Loss: 1.9176, Dev Loss: 1.8993, Dev Loss confidence interval: 1.8771, 1.9214
Ste

In [ ]:
model = LSTMProbNNDist(context_window, StudentTHead(), device)
train_path = "Train_StudentT/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)


Initializing LSTM with context_window=60, dist=StudentTHead, dof=3
num batches 81
Init, Train Loss: 1.8185, Dev Loss: 1.8130, Dev Loss confidence interval: 1.8054, 1.8205
Step 0, Train Loss: 1.7554, Dev Loss: 1.7487, Dev Loss confidence interval: 1.7398, 1.7577
Step 10, Train Loss: 1.7556, Dev Loss: 1.7490, Dev Loss confidence interval: 1.7401, 1.7579
Step 20, Train Loss: 1.7554, Dev Loss: 1.7488, Dev Loss confidence interval: 1.7398, 1.7577
Step 30, Train Loss: 1.7554, Dev Loss: 1.7487, Dev Loss confidence interval: 1.7398, 1.7577


In [6]:
model = LSTMProbNNDist(context_window, SkewedStudentTHead(), device)
train_path = "Train_SkewedStudentT/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)


Initializing LSTM with context_window=60, dist=SkewedStudentTHead, dof=4
num batches 81
Init, Train Loss: 1.6522, Dev Loss: 1.6475, Dev Loss confidence interval: 1.6408, 1.6542
Step 0, Train Loss: 1.6247, Dev Loss: 1.6199, Dev Loss confidence interval: 1.6126, 1.6271
Step 10, Train Loss: 1.6248, Dev Loss: 1.6200, Dev Loss confidence interval: 1.6127, 1.6272
Step 20, Train Loss: 1.6245, Dev Loss: 1.6196, Dev Loss confidence interval: 1.6124, 1.6269
Step 30, Train Loss: 1.6246, Dev Loss: 1.6198, Dev Loss confidence interval: 1.6125, 1.6271
Step 40, Train Loss: 1.6246, Dev Loss: 1.6198, Dev Loss confidence interval: 1.6125, 1.6272
Step 50, Train Loss: 1.6246, Dev Loss: 1.6198, Dev Loss confidence interval: 1.6125, 1.6271
Step 60, Train Loss: 1.6247, Dev Loss: 1.6199, Dev Loss confidence interval: 1.6126, 1.6272
Step 70, Train Loss: 1.6246, Dev Loss: 1.6208, Dev Loss confidence interval: 1.6136, 1.6281
Step 80, Train Loss: 1.6245, Dev Loss: 1.6198, Dev Loss confidence interval: 1.6125, 1.

In [ ]:
dof = 16
model = LSTMProbNNDist(context_window, BLogistic(dof - 2, device), device)
train_path = "Train_BLogistic/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)